# PR Governance Agent - Live Demo

**Zero-Touch & Zero-Trace Demo.**
This notebook is configured to use **Colab Secrets** for maximum security. 

### 🔑 Setup Instructions (One-time setup):
1. Click the **Key icon (🔑)** on the left sidebar.
2. Add these 5 Secrets and **enable notebook access** for each:
   - `GITHUB_TOKEN` (Your PAT)
   - `GATEWAY_URL` (Direct Agent URL)
   - `DEMO_REPO` (e.g., `lmudu2/risk-analyzer-poc`)
   - `JIRA_DOMAIN` (e.g., `name.atlassian.net`)
   - `APPROVER_EMAIL` (For display mask)

Once set, just click **Runtime > Run all**.

## Step 1: Initialize Environment

In [ ]:
import urllib.request, json, time, re, base64
from google.colab import userdata

def get_secret(key, default=None):
    try: 
        val = userdata.get(key)
        return val.strip() if val else default
    except: return default

try:
    # --- 1. Load Secrets with Fallbacks ---
    GITHUB_TOKEN = get_secret('GITHUB_TOKEN')
    GATEWAY_URL = get_secret('GATEWAY_URL')
    REPO = get_secret('DEMO_REPO') or get_secret('REPO')
    JIRA_DOMAIN = get_secret('JIRA_DOMAIN')
    APPROVER_EMAIL = get_secret('APPROVER_EMAIL')

    if not all([GITHUB_TOKEN, GATEWAY_URL, REPO]):
        missing = [k for k, v in {'GITHUB_TOKEN':GITHUB_TOKEN, 'GATEWAY_URL':GATEWAY_URL, 'DEMO_REPO':REPO}.items() if not v]
        raise Exception(f"Missing required Secrets in Colab sidebar: {', '.join(missing)}")
        
    GH_HEADERS = {
        "Authorization": f"token {GITHUB_TOKEN}",
        "Accept": "application/vnd.github.v3+json",
        "Content-Type": "application/json",
        "User-Agent": "PR-Agent-Demo"
    }
    
    def gh_url(path):
        path = path.strip('/')
        return f"https://api.github.com/repos/{REPO}/{path}" if path else f"https://api.github.com/repos/{REPO}"

    def gh_get(path):
        req = urllib.request.Request(gh_url(path), headers=GH_HEADERS)
        with urllib.request.urlopen(req, timeout=10) as r:
            return json.loads(r.read())
    
    def gh_post(path, data):
        req = urllib.request.Request(gh_url(path), data=json.dumps(data).encode(), headers=GH_HEADERS, method='POST')
        with urllib.request.urlopen(req, timeout=15) as r:
            return json.loads(r.read())

    print(f"\n✅ Configured Repo: {REPO}")
    print(f"✅ Token Active: {GITHUB_TOKEN[:5]}...{GITHUB_TOKEN[-4:] if (GITHUB_TOKEN and len(GITHUB_TOKEN)>10) else ''}")
    print(f"✅ Gateway Verified")
    print("\nReady for Zero-Touch Live Demo!")

except Exception as e:
    print(f"\u274c SETUP ERROR: {e}")
    print("\n1. Check the Key (🔑) icon on the left.\n2. Re-enter the values and TOGGLE ON 'Notebook access'.")

## Step 2: Auto-Create Test PR

In [ ]:
try:
    # Get base info
    repo_info = gh_get("")
    default_branch = repo_info["default_branch"]
    branch_info = gh_get(f"git/refs/heads/{default_branch}")
    base_sha = branch_info["object"]["sha"]

    # Create and commit
    branch_name = f"demo-agent-test-{int(time.time())}"
    gh_post("git/refs", {"ref": f"refs/heads/{branch_name}", "sha": base_sha})
    file_content = base64.b64encode(f"# Demo test file\nCreated at {time.strftime('%Y-%m-%d %H:%M:%S')}\n".encode()).decode()
    gh_post(f"contents/demo_test_{int(time.time())}.md", {
        "message": "demo: add test file",
        "content": file_content,
        "branch": branch_name
    })

    # Open PR
    pr = gh_post("pulls", {
        "title": f"[DEMO] Agent Audit - {time.strftime('%H:%M:%S')}",
        "head": branch_name,
        "base": default_branch,
        "body": "This PR was auto-created for the Live Agent Governance Demo."
    })
    PR_NUMBER = str(pr["number"])
    PR_URL = pr["html_url"]
    print(f"PR #{PR_NUMBER} created and ready: {PR_URL}")
except Exception as e:
    print(f"\u274c PR CREATION FAILED: {e}")
    if '404' in str(e):
        print("\nTIP: 404 usually means the REPO name is typed wrong or the TOKEN doesn't have access.")
    elif '401' in str(e):
        print("\nTIP: 401 means the TOKEN is expired or invalid.")

## Step 3: Trigger High Risk Analysis

In [ ]:
high_risk = {
    "action": "opened",
    "pull_request": {
        "number": int(PR_NUMBER),
        "title": "CRITICAL: delete user_access table",
        "user": {"login": "demo-presenter", "type": "User"}
    },
    "repository": {"full_name": REPO},
    "installation": {"id": 12345678},
    "sender": {"login": "demo-presenter", "type": "User"}
}
req = urllib.request.Request(GATEWAY_URL, data=json.dumps(high_risk).encode(),
    headers={"Content-Type": "application/json", "X-GitHub-Event": "pull_request"}, method='POST')
with urllib.request.urlopen(req, timeout=30) as res:
    print(f"Analysis triggered successfully. Agent is processing payload...")

## Step 4: View Live Results

In [ ]:
print("Waiting 20 seconds for processing...")
time.sleep(20)

pr_data = gh_get(f"pulls/{PR_NUMBER}")
state = pr_data.get('state','?').upper()
merged = pr_data.get('merged', False)
icon = 'MERGED' if merged else ('CLOSED - Blocked by Agent' if state=='CLOSED' else 'OPEN')
print(f"\n=== PR STATUS ===")
print(f"PR #{PR_NUMBER}: {icon}")

comments = gh_get(f"issues/{PR_NUMBER}/comments")
jira = None
print(f"\n=== AI RISK REPORT ===")
for c in reversed(comments or []):
    body = c.get('body','')
    if 'RISK' in body.upper():
        print(body[:800] + "...")
        m = re.search(r'([A-Z]+-\d+)', body)
        if m: jira = m.group(1)
        break
else:
    print("No report yet - please wait and re-run.")

print(f"\n=== JIRA TICKET ===")
if jira and JIRA_DOMAIN:
    print(f"Created: {jira}")
    print(f"URL: https://{JIRA_DOMAIN}/browse/{jira}")
else:
    print("No ticket created (or JIRA_DOMAIN secret not set).")

print(f"\n=== EMAIL NOTIFICATION ===")
if state == 'CLOSED' and not merged and APPROVER_EMAIL:
    mask_email = f"{APPROVER_EMAIL[:3]}...@{APPROVER_EMAIL.split('@')[-1]}"
    print(f"Approval email sent to {mask_email}")
else:
    print("No email required for this flow.")